# Web Scraping and Analysis of Books to Scrape

## 1. Scraping Book Details

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'http://books.toscrape.com/'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

books = []
for book in soup.find_all('article', class_='product_pod'):
    h3_tag = book.h3
    title_tag = h3_tag.a if h3_tag else None
    title = title_tag['title'] if title_tag else 'No Title'

    price_tag = book.find('p', class_='price_color')
    price = float(price_tag.text[2:]) if price_tag else 0.0

    rating_tag = book.p
    rating = rating_tag['class'][1] if rating_tag and len(rating_tag['class']) > 1 else 'No Rating'

    availability_tag = book.find('p', class_='instock availability')
    availability = availability_tag.text.strip() if availability_tag else 'Not Available'
    
    books.append({'Title': title, 'Price': price, 'Rating': rating, 'Availability': availability})

df = pd.DataFrame(books)
df.to_csv('books_scraped.csv', index=False)

## 2. Data Cleaning and Feature Engineering

In [ ]:
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df['Rating'] = df['Rating'].map(rating_map)

df['Affordability'] = df['Price'].apply(lambda x: 'Affordable' if x < 20 else 'Expensive')

average_rating = df.groupby('Affordability')['Rating'].mean().reset_index()
print(average_rating)

## 3. Data Visualization

In [ ]:
import matplotlib.pyplot as plt

affordability_counts = df['Affordability'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(affordability_counts, labels=affordability_counts.index.tolist(), autopct='%1.1f%%', startangle=140)
plt.title('Proportion of Affordable vs. Expensive Books')
plt.show()

plt.figure(figsize=(8, 6))
plt.bar(average_rating['Affordability'], average_rating['Rating'], color=['green', 'red'])
plt.xlabel('Affordability')
plt.ylabel('Average Rating')
plt.title('Average Ratings by Affordability')
plt.show()